# Fine-Tuning FunctionGemma for Ranga (Research Baseline)

> **Colab research notebook** — not deployed on phone. For production Android, use `ranga_gemma4_e2b_production.ipynb`.

[Fine-tuning with FunctionGemma](https://ai.google.dev/gemma/docs/functiongemma/finetuning-with-functiongemma)

## How to run in Google Colab

1. In **Google Drive**, go to `My Drive/capstone/training/notebooks/`
2. Right-click this notebook → **Open with → Google Colaboratory**
3. **Runtime → Change runtime type → T4 GPU** (or L4/A100 for E2B production)
4. Left sidebar **🔑 Secrets** → add `HF_TOKEN` (your [Hugging Face token](https://huggingface.co/settings/tokens))
5. Run **all cells in order** from top to bottom
6. After the **Install** cell finishes → **Runtime → Restart session** → run from **Colab setup** onward

### Required Drive folders

```
My Drive/capstone/
├── training/          ← this notebook + src/ranga_train/
└── dataset/
    └── ranga_output/  ← ranga_sft_500.jsonl, ranga_eval_50.jsonl, ranga_tools.json
```

In [ ]:
# @title 1. Install dependencies
# Run once, then: Runtime -> Restart session, and continue from "Colab setup".
!pip install -U torch tensorboard "transformers>=4.52" datasets accelerate evaluate trl protobuf sentencepiece matplotlib pandas ipywidgets --quiet

In [ ]:
# @title Colab setup — mount Drive, auth, GPU, paths
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from huggingface_hub import login

CAPSTONE_ROOT = "/content/drive/MyDrive/capstone"  # @param {type:"string"}

# 1) Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

CAPSTONE = Path(CAPSTONE_ROOT)
assert CAPSTONE.exists(), f"Drive folder not found: {CAPSTONE}"

# 2) Hugging Face token (Colab secret)
from google.colab import userdata
hf_token = userdata.get("HF_TOKEN")
login(token=hf_token)

# 3) GPU check
assert torch.cuda.is_available(), "Enable GPU: Runtime -> Change runtime type -> GPU"
print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")

# 4) Import project code from Drive
TRAINING_DIR = CAPSTONE / "training"
sys.path.insert(0, str(TRAINING_DIR / "src"))

from ranga_train.colab_paths import require_dataset, resolve_project_paths
from ranga_train.data import load_jsonl, prepare_eval_records, summarize_dataset
from ranga_train.evaluate import evaluate_model
from ranga_train.inference import make_generate_fn

paths = resolve_project_paths(capstone_root=CAPSTONE)
require_dataset(paths)

TRAINING_DIR = paths["training_dir"]
DATASET_DIR = paths["dataset_dir"]
REPO_ROOT = paths["repo_root"]
REPORTS_DIR = TRAINING_DIR / "results" / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(TRAINING_DIR)

sft_records = load_jsonl(DATASET_DIR / "ranga_sft_500.jsonl")
eval_records = prepare_eval_records(
    DATASET_DIR / "ranga_eval_50.jsonl",
    DATASET_DIR / "ranga_tools.json",
)
print("Capstone root:", REPO_ROOT)
print("Training dir:", TRAINING_DIR)
print("Dataset dir:", DATASET_DIR)
print("Dataset summary:", summarize_dataset(sft_records))
print("Held-out eval queries:", len(eval_records))

## Prepare FunctionGemma dataset

Maps `system` → `developer` and compacts tool payloads for GPU memory.

In [ ]:
# @title 2. Build train/validation split
from ranga_train.config import TrainConfig
from ranga_train.data import prepare_sft_dataset
from ranga_train.train import build_model_and_tokenizer, build_trainer, save_training_curves

config = TrainConfig(
    dataset_dir=DATASET_DIR,
    checkpoint_dir=TRAINING_DIR / "results" / "functiongemma_checkpoints",
    reports_dir=REPORTS_DIR,
)
config.checkpoint_dir.mkdir(parents=True, exist_ok=True)

hf_datasets = prepare_sft_dataset(
    config.sft_path,
    train_split=config.train_split,
    seed=config.seed,
    max_hospitals=config.max_hospitals_in_tool_payload,
    role_style="functiongemma",
)
print(f"Train={len(hf_datasets['train'])}  Val={len(hf_datasets['validation'])}")

SYSTEM_PROMPT = next(m["content"] for m in sft_records[0]["messages"] if m["role"] == "system")
PROMPT_ROLE = "developer"

## Baseline evaluation (pre fine-tuning)

In [ ]:
# @title 3. Load base model + baseline eval
baseline_model, baseline_tokenizer = build_model_and_tokenizer(config, token=hf_token)
baseline_generate = make_generate_fn(baseline_model, baseline_tokenizer)

baseline_report = evaluate_model(
    model_label="functiongemma_baseline",
    eval_records=eval_records,
    system_prompt=SYSTEM_PROMPT,
    generate_fn=baseline_generate,
    prompt_role=PROMPT_ROLE,
)
pd.DataFrame([baseline_report.to_dict()])

## Supervised fine-tuning

In [ ]:
# @title 4. Train + loss curve
trainer = build_trainer(config, baseline_model, baseline_tokenizer, hf_datasets)
trainer.train()
trainer.save_model()
curves = save_training_curves(trainer, REPORTS_DIR / "functiongemma_training_curves.json")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(curves["epoch_train"], curves["train_loss"], label="Train")
ax.plot(curves["epoch_eval"], curves["eval_loss"], label="Validation")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.legend(); ax.grid(True, alpha=0.3)
plt.savefig(REPORTS_DIR / "functiongemma_loss_curve.png", dpi=150)
plt.show()

## Post fine-tuning evaluation

In [ ]:
# @title 5. Evaluate fine-tuned model
from dataclasses import replace

ft_config = replace(config, base_model=str(config.checkpoint_dir))
finetuned_model, finetuned_tokenizer = build_model_and_tokenizer(ft_config, token=hf_token)
finetuned_generate = make_generate_fn(finetuned_model, finetuned_tokenizer)

finetuned_report = evaluate_model(
    model_label="functiongemma_finetuned",
    eval_records=eval_records,
    system_prompt=SYSTEM_PROMPT,
    generate_fn=finetuned_generate,
    prompt_role=PROMPT_ROLE,
)
comparison_df = pd.DataFrame(finetuned_report.comparison_table(baseline_report))
comparison_df

In [ ]:
# @title 6. Save reports to Drive
(REPORTS_DIR / "functiongemma_baseline.json").write_text(json.dumps(baseline_report.to_dict(), indent=2))
(REPORTS_DIR / "functiongemma_finetuned.json").write_text(json.dumps(finetuned_report.to_dict(), indent=2))
comparison_df.to_csv(REPORTS_DIR / "functiongemma_comparison.csv", index=False)
print("Saved to", REPORTS_DIR)

## Next: production on phone

Open `ranga_gemma4_e2b_production.ipynb` → fine-tune Gemma 4 E2B → export `.litertlm` for Flutter.